# Enhanced Traffic Management System Analysis

This notebook analyzes the performance and results of the Enhanced Traffic Management System, a reinforcement learning-based approach to traffic signal optimization for Indian urban conditions.

## 1. Import Required Libraries

First, let's import the necessary libraries for our analysis.

In [ ]:
# Import essential libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
import json

# For working with the traffic environment
try:
    import gymnasium as gym
    import torch
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
    SB3_AVAILABLE = True
except ImportError:
    print("Warning: stable-baselines3 or gymnasium not available. Some features will be disabled.")
    SB3_AVAILABLE = False

# Set plotting style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

# Configure paths
base_path = Path("c:/Users/Lenovo/Documents/GitHub/stms-backend")
logs_path = base_path / "logs"
models_path = base_path / "models"

print("Libraries imported successfully")

## 2. Load and Explore Training Data

Now let's load the training logs from our reinforcement learning model and explore the learning progress.

In [ ]:
# Function to check and load training logs
def check_logs():
    """Check if training logs exist and return a list of log files"""
    if os.path.exists(logs_path):
        return [f for f in os.listdir(logs_path) if f.endswith('.monitor.csv')]
    return []

# Get list of log files
log_files = check_logs()
print(f"Found log files: {log_files}")

# Function to load training data
def load_training_data():
    """Load and combine training data from all log files"""
    data = []
    for log_file in log_files:
        file_path = os.path.join(logs_path, log_file)
        try:
            # Skip header comments in monitor files
            df = pd.read_csv(file_path, skiprows=1)
            df['source'] = log_file  # Add source file for reference
            data.append(df)
            print(f"Loaded {len(df)} rows from {log_file}")
        except Exception as e:
            print(f"Error loading {log_file}: {e}")
    
    if data:
        return pd.concat(data)
    else:
        print("No training data available")
        return None

# Load the data
training_data = load_training_data()

# Display the first few rows if data is available
if training_data is not None:
    print("\nTraining data preview:")
    display(training_data.head())
    
    # Show basic statistics
    print("\nTraining data statistics:")
    display(training_data.describe())
else:
    print("No data available for analysis. Please run the training first.")

## 3. Visualize Learning Progress

Let's visualize how the reinforcement learning agent improved over time by plotting the reward and episode length trends.

In [ ]:
# Plot learning curves if training data is available
if training_data is not None:
    # Create a figure with two subplots for rewards and episode lengths
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # Plot rewards over time
    ax1.plot(training_data['r'], label='Episode Reward', color='blue', alpha=0.6)
    ax1.plot(training_data['r'].rolling(window=10).mean(), 
             label='10-Episode Moving Average', color='darkblue', linewidth=2)
    ax1.set_ylabel('Reward')
    ax1.set_title('Training Rewards Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot episode lengths over time
    ax2.plot(training_data['l'], label='Episode Length', color='green', alpha=0.6)
    ax2.plot(training_data['l'].rolling(window=10).mean(), 
             label='10-Episode Moving Average', color='darkgreen', linewidth=2)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Episode Length')
    ax2.set_title('Episode Lengths Over Time')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Create additional visualizations for deeper analysis
    
    # Histogram of rewards
    plt.figure(figsize=(12, 6))
    sns.histplot(data=training_data, x='r', kde=True, bins=30)
    plt.title('Distribution of Episode Rewards')
    plt.xlabel('Reward')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Reward vs Episode Length
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=training_data, x='l', y='r', alpha=0.6)
    plt.title('Reward vs Episode Length')
    plt.xlabel('Episode Length')
    plt.ylabel('Reward')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No training data available for visualization.")

## 4. Load and Test the Trained Model

Let's load the trained PPO model and test it on different traffic scenarios.

In [ ]:
# Import the traffic environment
if SB3_AVAILABLE:
    # Check for model file
    model_path = models_path / "traffic_model.zip"
    if model_path.exists():
        print(f"Model found at: {model_path}")
        
        # We need to import our environment to load the model
        import sys
        sys.path.append(str(base_path))
        
        try:
            # Import the environment
            from fixed_traffic_system import EnhancedTrafficEnv
            
            # Create the environment
            env = EnhancedTrafficEnv(
                num_intersections=1,
                num_roads_per_intersection=4,
                max_vehicles=50,
                max_steps=100,
                use_llm_features=False
            )
            
            # Wrap the environment as done during training
            from stable_baselines3.common.monitor import Monitor
            env = Monitor(env)
            vec_env = DummyVecEnv([lambda: env])
            
            # Try to load the model
            try:
                model = PPO.load(model_path, env=vec_env)
                print("Model loaded successfully!")
            except Exception as e:
                print(f"Error loading model: {e}")
                model = None
        except Exception as e:
            print(f"Error importing or creating environment: {e}")
            env = None
            model = None
    else:
        print(f"Model not found at: {model_path}")
        env = None
        model = None
else:
    print("stable-baselines3 or gymnasium not available. Cannot load model.")
    env = None
    model = None

In [ ]:
# Function to evaluate the model on a specific traffic scenario
def evaluate_traffic_scenario(env, model, scenario_name, config, num_episodes=5):
    """
    Evaluate the model on a specific traffic scenario
    
    Args:
        env: The environment
        model: The trained PPO model
        scenario_name: Name of the scenario
        config: Configuration dict with weather_impact, traffic_signs, incidents, pedestrians
        num_episodes: Number of episodes to evaluate
    
    Returns:
        Dict with evaluation metrics
    """
    if env is None or model is None:
        print("Environment or model not available. Skipping evaluation.")
        return None
    
    rewards = []
    episode_lengths = []
    waiting_times = []
    
    print(f"\nEvaluating scenario: {scenario_name}")
    print(f"Configuration: {config}")
    
    for ep in range(num_episodes):
        obs, _ = env.envs[0].env.reset()
        unwrapped_env = env.envs[0].env.unwrapped
        
        # Configure environment according to scenario
        unwrapped_env.set_weather_impact(config["weather_impact"])
        
        for sign in config["traffic_signs"]:
            unwrapped_env.add_traffic_sign(sign)
            
        if config["incidents"]:
            unwrapped_env.add_traffic_incident(0, 1, severity=0.8)
            
        if config["pedestrians"]:
            unwrapped_env.pedestrian_presence[0, 2] = 0.9
        
        # Run one episode
        episode_reward = 0
        episode_length = 0
        done = False
        truncated = False
        
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            
            episode_reward += reward
            episode_length += 1
            
            done = terminated[0]
            truncated = truncated[0]
            
            # Extract waiting time info if available
            if isinstance(info, dict) and 0 in info and 'cumulative_waiting' in info[0]:
                cumulative_waiting = info[0]['cumulative_waiting']
        
        rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        
        print(f"  Episode {ep+1}: Reward={episode_reward:.2f}, Length={episode_length}")
    
    # Calculate statistics
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)
    mean_length = np.mean(episode_lengths)
    
    print(f"  Results: Mean reward={mean_reward:.2f}±{std_reward:.2f}, Mean length={mean_length:.2f}")
    
    return {
        "name": scenario_name,
        "reward": mean_reward,
        "reward_std": std_reward,
        "episode_length": mean_length,
        "config": config
    }

# Skip this if env or model is not available
if env is not None and model is not None:
    print("Ready to evaluate different traffic scenarios.")
else:
    print("Cannot evaluate scenarios without environment and model.")

## 5. Evaluate Different Traffic Scenarios

Let's test how our model performs across various traffic conditions typical for Indian urban environments.

In [ ]:
# Define different traffic scenarios to test
scenarios = {
    "normal": {
        "weather_impact": 1.0,
        "traffic_signs": [],
        "incidents": False,
        "pedestrians": False
    },
    "monsoon_rain": {
        "weather_impact": 0.6,
        "traffic_signs": [],
        "incidents": False,
        "pedestrians": False
    },
    "school_zone": {
        "weather_impact": 1.0,
        "traffic_signs": ["pedestrian_crossing"],
        "incidents": False,
        "pedestrians": True
    },
    "road_accident": {
        "weather_impact": 1.0,
        "traffic_signs": [],
        "incidents": True,
        "pedestrians": False
    },
    "festival_rush": {
        "weather_impact": 1.0,
        "traffic_signs": [],
        "incidents": False,
        "pedestrians": True
    },
    "complex_scenario": {
        "weather_impact": 0.7,
        "traffic_signs": ["hump", "pedestrian_crossing"],
        "incidents": True,
        "pedestrians": True
    }
}

# Run evaluations if environment and model are available
results = []

if env is not None and model is not None:
    for name, config in scenarios.items():
        result = evaluate_traffic_scenario(env, model, name, config, num_episodes=3)
        if result:
            results.append(result)
else:
    print("Skipping scenario evaluations due to missing environment or model.")

## 6. Visualize Performance Across Scenarios

Let's create visualizations to compare how our traffic management system performs across different scenarios.

In [ ]:
# Visualize scenario comparison if we have results
if results:
    # Convert results to DataFrame for easier visualization
    df_results = pd.DataFrame([
        {
            "Scenario": r["name"],
            "Mean Reward": r["reward"],
            "Reward Std": r["reward_std"],
            "Mean Episode Length": r["episode_length"],
            "Weather Impact": r["config"]["weather_impact"],
            "Traffic Signs": len(r["config"]["traffic_signs"]),
            "Incidents": int(r["config"]["incidents"]),
            "Pedestrians": int(r["config"]["pedestrians"])
        }
        for r in results
    ])
    
    # Display the results table
    print("Performance across scenarios:")
    display(df_results)
    
    # Create bar chart comparing rewards across scenarios
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(data=df_results, x="Scenario", y="Mean Reward", 
                    palette="viridis", errorbar=("ci", 95))
    
    # Add value labels on top of bars
    for i, bar in enumerate(ax.patches):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + (5 if bar.get_height() > 0 else -20),
            f'{bar.get_height():.1f}',
            ha='center',
            color='black',
            fontweight='bold'
        )
    
    plt.title("Performance Comparison Across Traffic Scenarios")
    plt.ylabel("Mean Reward")
    plt.xlabel("Scenario")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Create a radar chart to compare multiple metrics across scenarios
    # Prepare the data
    categories = list(df_results['Scenario'])
    N = len(categories)
    
    # Create normalized values for different metrics (0-1 scale)
    # For reward, higher is better, but we need to normalize it to a 0-1 scale
    min_reward = df_results['Mean Reward'].min()
    max_reward = df_results['Mean Reward'].max()
    reward_range = max_reward - min_reward
    
    if reward_range == 0:  # Avoid division by zero
        normalized_rewards = [0.5] * N
    else:
        normalized_rewards = [(r - min_reward) / reward_range for r in df_results['Mean Reward']]
    
    # For episode length, we'll consider shorter episodes better
    min_length = df_results['Mean Episode Length'].min()
    max_length = df_results['Mean Episode Length'].max()
    length_range = max_length - min_length
    
    if length_range == 0:  # Avoid division by zero
        normalized_lengths = [0.5] * N
    else:
        normalized_lengths = [1 - ((l - min_length) / length_range) for l in df_results['Mean Episode Length']]
    
    # Weather impact (higher is better)
    weather_impacts = df_results['Weather Impact'].values
    
    # Complexity score (lower is better) - combined effect of signs, incidents, pedestrians
    complexity_scores = [
        1 - ((df_results['Traffic Signs'][i] + df_results['Incidents'][i] + df_results['Pedestrians'][i]) / 5)
        for i in range(len(df_results))
    ]
    
    # Combine all metrics
    values = np.array([
        normalized_rewards,
        normalized_lengths,
        weather_impacts,
        complexity_scores
    ])
    
    # Create the radar chart
    metrics = ['Reward', 'Efficiency', 'Weather Handling', 'Complexity Handling']
    
    # Set up the radar chart
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # Close the loop
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    
    # Plot each scenario
    for i, scenario in enumerate(categories):
        values_scenario = values[:, i].tolist()
        values_scenario += values_scenario[:1]  # Close the loop
        
        ax.plot(angles, values_scenario, linewidth=2, label=scenario)
        ax.fill(angles, values_scenario, alpha=0.1)
    
    # Set category labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    
    # Add legend
    ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    
    plt.title('Performance Metrics Across Different Scenarios', size=15)
    plt.tight_layout()
    plt.show()
else:
    print("No scenario results available for visualization.")

## 7. Traffic Pattern Analysis

Let's analyze the traffic patterns from our simulation to understand how our traffic management system responds to different conditions.

In [ ]:
# Function to simulate and record traffic patterns
def simulate_traffic_patterns(env, model, num_steps=200):
    """
    Run a simulation and record detailed traffic patterns
    
    Args:
        env: The environment
        model: The trained model
        num_steps: Number of steps to simulate
    
    Returns:
        DataFrame with traffic data
    """
    if env is None or model is None:
        print("Environment or model not available. Cannot simulate traffic patterns.")
        return None
    
    # Reset the environment
    obs, _ = env.envs[0].env.reset()
    unwrapped_env = env.envs[0].env.unwrapped
    
    # Prepare data collection
    data = []
    
    # Run simulation
    for step in range(num_steps):
        # Get action from model
        action, _ = model.predict(obs, deterministic=True)
        
        # Take step in environment
        obs, reward, terminated, truncated, info = env.step(action)
        
        # Extract traffic data
        vehicles = unwrapped_env.vehicles.copy()
        current_phase = unwrapped_env.current_phase.copy()
        is_yellow = unwrapped_env.is_yellow.copy()
        waiting_times = unwrapped_env.waiting_times.copy()
        
        # Record data for this step
        data.append({
            'step': step,
            'action': action[0] if isinstance(action, np.ndarray) else action,
            'reward': reward[0],
            'vehicles': vehicles.copy(),
            'current_phase': current_phase.copy(),
            'is_yellow': is_yellow.copy(),
            'waiting_times': waiting_times.copy(),
            'total_vehicles': np.sum(vehicles),
            'max_waiting': np.max(waiting_times),
            'total_waiting': np.sum(waiting_times)
        })
        
        # Check if episode ended
        if terminated[0] or truncated[0]:
            break
    
    print(f"Simulated {len(data)} steps")
    return data

# Run simulation if environment and model are available
if env is not None and model is not None:
    print("Running traffic pattern simulation...")
    traffic_data = simulate_traffic_patterns(env, model, num_steps=150)
    
    if traffic_data:
        # Extract key metrics for plotting
        steps = [d['step'] for d in traffic_data]
        total_vehicles = [d['total_vehicles'] for d in traffic_data]
        total_waiting = [d['total_waiting'] for d in traffic_data]
        actions = [d['action'] for d in traffic_data]
        
        # Plot traffic patterns
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
        
        # Plot total vehicles
        ax1.plot(steps, total_vehicles, 'b-', label='Total Vehicles')
        ax1.set_ylabel('Number of Vehicles')
        ax1.set_title('Traffic Volume Over Time')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot waiting times
        ax2.plot(steps, total_waiting, 'r-', label='Total Waiting Time')
        ax2.set_ylabel('Waiting Time')
        ax2.set_title('Traffic Congestion Over Time')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot actions (traffic signal changes)
        ax3.step(steps, actions, 'g-', where='post', label='Traffic Signal Phase')
        ax3.set_ylabel('Signal Phase')
        ax3.set_xlabel('Simulation Step')
        ax3.set_title('Traffic Signal Control Decisions')
        ax3.set_yticks(range(4))
        ax3.set_yticklabels(['Road 0', 'Road 1', 'Road 2', 'Road 3'])
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Create heatmap of vehicle distribution and waiting times
        if len(traffic_data) > 0:
            # Sample data points to avoid overcrowding
            sample_indices = np.linspace(0, len(traffic_data)-1, min(20, len(traffic_data)), dtype=int)
            sample_data = [traffic_data[i] for i in sample_indices]
            
            # Extract vehicle counts per road
            vehicle_data = np.array([list(d['vehicles'][0]) for d in sample_data])
            
            # Create heatmap
            plt.figure(figsize=(12, 6))
            sns.heatmap(vehicle_data.T, cmap="YlGnBu", 
                        xticklabels=[d['step'] for d in sample_data],
                        yticklabels=['Road 0', 'Road 1', 'Road 2', 'Road 3'])
            plt.title('Vehicle Distribution Across Roads Over Time')
            plt.xlabel('Simulation Step')
            plt.ylabel('Road')
            plt.tight_layout()
            plt.show()
    else:
        print("No traffic data generated from simulation.")
else:
    print("Skipping traffic pattern simulation due to missing environment or model.")

## 8. Conclusion and Future Work

Let's summarize our findings and discuss potential improvements for the traffic management system.

### Summary of Findings

Our analysis of the Enhanced Traffic Management System has provided several insights:

1. **Training Performance**: The PPO reinforcement learning algorithm shows improvement over time as it learns effective traffic signal control strategies.

2. **Scenario Adaptation**: The model demonstrates varying performance across different traffic scenarios, with particularly strong or weak responses to specific conditions.

3. **Traffic Patterns**: The system's decisions show clear correlations with traffic volumes and waiting times, suggesting appropriate responsiveness to changing conditions.

4. **Handling Special Conditions**: The model adapts to adverse weather, pedestrian presence, traffic signs, and incidents, though with varying degrees of effectiveness.

### Strengths of the Current System

- **Adaptability**: The system can handle a wide range of traffic conditions.
- **Modular Design**: The architecture allows for easy integration of new features and detectors.
- **Indian Traffic Context**: Specifically designed for Indian urban traffic characteristics.
- **Reinforcement Learning**: Continuously improves through experience.

### Areas for Improvement

1. **Extended Training**: The model would benefit from longer training with more diverse scenarios.

2. **Multi-Intersection Coordination**: Enhance the coordination between multiple intersections.

3. **Real-World Validation**: Test with real traffic data from Indian cities.

4. **LLM Integration**: Improve the use of LLM insights for strategic decision-making.

5. **Visual Interface**: Develop a more intuitive visualization system for traffic operators.

### Next Steps

1. **Data Collection**: Gather real traffic data from Indian urban intersections.

2. **Algorithm Comparison**: Compare PPO with other RL algorithms like A3C, SAC, or DQN.

3. **Hardware Integration**: Connect with actual traffic cameras and signals.

4. **Mobile App**: Develop a mobile interface for traffic police and operators.

5. **Distributed System**: Scale to city-wide deployment with edge computing.